In [42]:
# This code is part of QCMet.
# 
# (C) Copyright 2024 National Physical Laboratory and National Quantum Computing Centre 
# 
# Licensed under the Apache License, Version 2.0 (the "License");
# you may not use this file except in compliance with the License.
# You may obtain a copy of the License at
# 
#      http://www.apache.org/licenses/LICENSE-2.0
# 
# Unless required by applicable law or agreed to in writing, software
# distributed under the License is distributed on an "AS IS" BASIS,
# WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
# See the License for the specific language governing permissions and
# limitations under the License


# VQE Benchmark using the 1D Fermi Hubbard model

## Hubbard Hamiltonian
The 1D Hubbard model Hamiltonian is given by 
\begin{equation*}
\hat{H} = -t \sum_{i=1}^{N-1}\sum_{\sigma \in \{\uparrow,\downarrow\}} \left(\hat{c}_{i\sigma}^\dagger\hat{c}_{i+1\sigma} +\hat{c}_{i+1\sigma}^\dagger\hat{c}_{i\sigma} \right) + U \sum_{i=1}^{N} \hat{n}_{i\uparrow}\hat{n}_{i\downarrow},
\end{equation*}
where $\hat{c}^\dagger_{i\sigma}$ ($\hat{c}_{i\sigma}$) is the creation (annihilation) operator for a Fermion in site $i$ with spin $\sigma$, $\hat{n}_{i\sigma}$ is the corresponding number operator, $t$ is the hopping integral, and $U$ is the onsite energy. Typically $t$ is taken to be positive, whereas $U$ can be positive (repulsive interactions) or negative (attractive interactions).

## Fermion to Spin Mapping
In order to obtain a quantum circuit representation of the evolution operator for the Hubbard model, it is first necessary to map the fermionic system onto a system of qubits.  Here we have a total of $2N$ distinct Fermionic sites ($N$ physical sites each with $2$ spin sites), here will map this onto a system of $2N$ qubits using a Jordan-Wigner mapping.  In order to do so it is first necessary to specify the ordering of the spin sites, here we will use the following ordering of spins into an up and down chain
\begin{equation*}
    \{1\uparrow, 2\uparrow, \dots, N\uparrow, 1\downarrow, 2\downarrow, \dots N\downarrow\} \rightarrow \{1, 2, \dots N, N+1, N+2 \dots, 2N\}.
\end{equation*}
With this ordering and the use of the Jordan-Wigner mapping the Fermionic Hamiltonian maps onto a qubit Hamiltonian of the form 
\begin{align*}
    \hat{H}_q &= -t \sum_{\substack{i=1 \\ i\neq N}}^{2N-1} \left(\hat{\sigma}_i^+ \hat{\sigma}_{i+1}^- + \hat{\sigma}_{i+1}^+ \hat{\sigma}_{i}^-\right) + \frac{U}{4} \sum_{i=1}^{N} \left(-\hat{\sigma}_{i}^z + \hat{1}\right)\left(-\hat{\sigma}_{i+N}^z + \hat{1}\right) \\
    &= -t \sum_{\substack{i=1 \\ i\neq N}}^{2N-1} \left(\hat{\sigma}_i^+ \hat{\sigma}_{i+1}^- + \hat{\sigma}_{i+1}^+ \hat{\sigma}_{i}^-\right) + \frac{U}{4} \sum_{i=1}^{N}\hat{\sigma}_{i}^z\hat{\sigma}_{i+N}^z - \frac{U}{4}\sum_{i=1}^{2N} \hat{\sigma}_i^z + \frac{NU}{4}, 
\end{align*}
where $\hat{\sigma}_i^\pm = \frac{1}{2}\left(\hat{\sigma}_i^{x} \pm i \hat{\sigma}_i^{y}\right)$ are the qubit $i$ raising ($+$) and lowering ($-$) operators. The constant energy shift in the last line simply corresponds to an arbitrary phase rotation and so can be ignored.  Consequently we can consider the qubit Hamiltonian
\begin{align*}
    \hat{H} &= -t \sum_{\substack{i=1 \\ i\neq N}}^{2N-1} \left(\hat{\sigma}_i^+ \hat{\sigma}_{i+1}^- + \hat{\sigma}_{i+1}^+ \hat{\sigma}_{i}^-\right) + \frac{U}{4} \sum_{i=1}^{N}\hat{\sigma}_{i}^z\hat{\sigma}_{i+N}^z - \frac{U}{4}\sum_{i=1}^{2N} \hat{\sigma}_i^z \\
    &= -\frac{t}{2} \sum_{\substack{i=1 \\ i\neq N}}^{2N-1} \left(\hat{\sigma}_i^x \hat{\sigma}_{i+1}^x + \hat{\sigma}_{i}^y \hat{\sigma}_{i+1}^y\right) + \frac{U}{4} \sum_{i=1}^{N}\hat{\sigma}_{i}^z\hat{\sigma}_{i+N}^z - \frac{U}{4}\sum_{i=1}^{2N} \hat{\sigma}_i^z,
\end{align*}
containing two-body hoping terms and two and one-body terms arising from the interaction.

## VQE
Here we consider the use of a Hamiltonian variational ansatz for the evaluation of the ground state properties of the 1D Fermi-Hubbard model, and will largely follow reference [1]. Here we will employ the same Fermion-to-Qubit mapping as described above, and employ as the Hamiltonian variational ansatz, which is a parametrised version of a trotterized propagator constructed from the Hamiltonian.

### Time Evolution: Trotterization of the Dynamics

The dynamics associated with this model is generated by the propagator
\begin{equation*}
    \hat{U} = e^{-i\hat{H}t} = \left[e^{-i\hat{H}\frac{t}{N}}\right]^N,
\end{equation*}
working in units where $\hbar=1$.  

Now in order to Trotterize we will proceed by Trotterizing, the short time propagators above by splitting the Hamiltonian into the two sets of mutually commuting hopping terms and the interaction terms
\begin{equation*}
    \hat{H}_q = \hat{H}_0+ \hat{H}_1 + \hat{H}_z.
\end{equation*}
Then we have
\begin{equation*}
    \hat{H}_j = -t \sum_{{i=1,\,\, i (mod \,2) = j}}^{N-1} \left(\hat{\sigma}_i^+ \hat{\sigma}_{i+1}^- + \hat{\sigma}_{i+1}^+ \hat{\sigma}_{i}^- + \hat{\sigma}_{i+N}^+ \hat{\sigma}_{i+N+1}^- + \hat{\sigma}_{i+N+1}^+ \hat{\sigma}_{i+N}^-\right), 
\end{equation*}
and 
\begin{equation*}
    \hat{H}_z = \frac{U}{4} \sum_{i=1}^{N}\hat{\sigma}_{i}^z\hat{\sigma}_{i+N}^z - \frac{U}{4}\sum_{i=1}^{2N} \hat{\sigma}_i^z.
\end{equation*}


The resultant Trotterized short time propagator can then be written as
\begin{equation}
\hat{U}_{\delta t}=e^{-i\hat{H}\delta t} \approx e^{-i\hat{H_0}\delta t} e^{-i\hat{H_1}\delta t}  e^{-i\hat{H_z}\delta t} = \hat{U}_0\hat{U}_1\hat{U}_Z,
\end{equation}
where 
\begin{align}
\hat{U}_j& = \prod_{{i=1,\,\, i (mod \,2) = j}}^{N-1} \exp\left[ i t\delta t \left(\hat{\sigma}_i^+ \hat{\sigma}_{i+1}^- + \hat{\sigma}_{i+1}^+ \hat{\sigma}_{i}^-\right)\right] \exp\left[ i t\delta t \left(\hat{\sigma}_{i+N}^+ \hat{\sigma}_{i+N+1}^- + \hat{\sigma}_{i+N+1}^+ \hat{\sigma}_{i+N}^-\right) \right] \\
& =\prod_{{i=1,\,\, i (mod \,2) = j}}^{N-1} \exp\left[ \frac{i t\delta t}{2} \left(\hat{\sigma}_i^x \hat{\sigma}_{i+1}^x + \hat{\sigma}_{i+1}^y \hat{\sigma}_{i}^y\right)\right] \exp\left[ \frac{i t\delta t}{2} \left(\hat{\sigma}_{i+N}^x \hat{\sigma}_{i+N+1}^x + \hat{\sigma}_{i+N}^y \hat{\sigma}_{i+N+1}^y\right) \right] \\
& =\prod_{{i=1,\,\, i (mod \,2) = j}}^{N-1} \left[R_{XX+YY}^{(i, i+1)}\left(-2t\delta t\right) R_{XX+YY}^{(i+N, i+N+1)}\left(-2t\delta t\right) \right],
\end{align}
consists of a layer of parallel 2 qubit unitaries, and 
\begin{align}
\hat{U}_z &= \prod_{i=1}^{N} \exp\left[-i\frac{U\delta t}{4} \hat{\sigma}_{i}^z\hat{\sigma}_{i+N}^z\right] \exp\left[i\frac{U\delta t}{4} \hat{\sigma}_{i}^z\right] \exp\left[i\frac{U\delta t}{4}\hat{\sigma}_{i+N}^z\right] \\
&=  \prod_{i=1}^{N} \left[R_{zz}^{(i, i+N)}\left(\frac{U\delta t}{2}\right) R_{z}^{(i)}\left(-\frac{U\delta t}{2}\right)R_{z}^{(2N-i)}\left(-\frac{U\delta t}{2}\right)\right],
\end{align}
consists of a layer of single qubit unitaries followed by a layer of parallel, non-local 2 qubit unitaries.


### Hamiltonian Variational Ansatz

Given the trotterized propagator from the Hamiltonian, the Hamiltonian variational ansatz can be written as
\begin{equation*}
\ket{\psi(\boldsymbol{\theta})} = \prod_{n=1}^{N_L} \left[e^{-i\hat{H_0}\theta_0^{(n)}} e^{-i\hat{H_1}\theta_1^{(n)}}  e^{-i\hat{H_z}\theta_z^{(n)}} \right] \ket{\psi_0},
\end{equation*}

where $\boldsymbol{\theta} = (\theta_0^{(1)}, \theta_1^{(1)}, \theta_z^{(1)}, \dots, \theta_0^{(N_L)}, \theta_1^{(N_L)}, \theta_z^{(N_L)})$ is the set of variational parameters to optimise, $N_L$ is the number of layers to be applied, and $\ket{\psi_0}$ is some initial product state.

## Ground State
In order to apply the VQE ansatz we need to prepare an initial state $\ket{\Psi_0}$.  Here we will make use of the ground state for the non-interacting, $U=0$, problem, although other choices such as the Hartree-Fock ground state are possible.  To do this we need to transform to a basis where the hopping terms is diagonal.  Namely, we will introduce a new set of ($k$-space) fermionic creation operators
\begin{equation}
\hat{c}_{k\sigma} = \frac{1}{\sqrt{N}}\sum_{j=1}^N e^{i k j a} \hat{c}_{j\sigma}, 
\end{equation}
with $k = 2\pi n/N$, $n=0,1\dots,N-1$, and $a$ a lattice spacing. Introducing these terms transforms the non-interacting fermionic Hamiltonian,
\begin{equation}
\hat{H} = -t \sum_{i=1}^{N-1}\sum_{\sigma \in \{\uparrow,\downarrow\}} \left(\hat{c}_{i\sigma}^\dagger\hat{c}_{i+1\sigma} +\hat{c}_{i+1\sigma}^\dagger\hat{c}_{i\sigma} \right),
\end{equation}
into
\begin{equation}
\hat{H} =  \sum_{n=0}^{N-1}\sum_{\sigma \in \{\uparrow,\downarrow\}} \epsilon_k \left(\hat{c}_{k\sigma}^\dagger\hat{c}_{k\sigma} \right).
\end{equation}
where $\epsilon_k = -2t \cos(ka)$.  The non-interacting ground state in $k$-space can then be obtained by applying all particles with $\epsilon_k \leq 0$, which in that is the state
\begin{equation}
\ket{\psi_0}_k = \prod_{\sigma \in \{\uparrow,\downarrow\}}  \prod_{k, \epsilon_k < 0} \hat{c}_{k\sigma}\ket{0}.
\end{equation}


### References

[1] Anselme Martin, B., Simon, P., & Rančić, M. J. (2022). Simulating strongly interacting Hubbard chains with the variational Hamiltonian ansatz on a quantum computer. Physical Review Research, 4(2), 023190.

#### Imports

In [44]:
import vqe
import numpy as np
import sys
import pathlib
import os
from tqdm.autonotebook import tqdm
sys.path.insert(0, str(pathlib.Path(os.path.abspath('')).parent.parent.parent.resolve()))

from _helpers.circuit_submitter import CircuitSubmitter


In [45]:
device_name = "noisy_sim"
submitter = CircuitSubmitter("vqe-fermi-hubbard", device_name)

In [46]:
def get_vqe_instance(nsites, U, num_layers_vha):
    """_summary_

    :param int nsites: Number of fermionic sites in Hamiltonian
    :param float U: the onsite energy
    :param int num_layers_vha: the number of layers of the ansatz to use
    :return FermiHubbardVQE: 
    """
    fhq = vqe.FermiHubbardVQE(nsites=nsites, U=U, shift_number=False)
    fhq.prepare_initial_state_qiskit()
    for _ in range(num_layers_vha):
        fhq.apply_vha()
    fhq.energy_expectation_operators()
    return fhq


# Parameters
nsites = 3  # num_qubits = 2*nsites
num_layers_vha = 1
U=2
shots=1000
num_trials = 100

# Seed to ensure repeatability
starting_seed = 5
np.random.seed(starting_seed)

vqe_experiments = []
energy_differences = []
for _ in range(num_trials):
    fhq = get_vqe_instance(nsites, U, num_layers_vha)
    x0 = np.random.random(fhq.params._size)
    vqe_experiments.append((fhq, x0))

for experiment, params  in tqdm(vqe_experiments):
    circ_list = []
    results= {}
    for key, circ in experiment.qc_list.items():
       circ_list.append(circ.assign_parameters(params).decompose(reps=2))
    tasks = submitter.submit_circuits(shots=shots, skip_asking=True, qasm_strs=[qc.qasm() for qc in circ_list], print_summary = False)
    counts = submitter.retrieve_counts([task.id for task in tasks], wait=True, print_timestamp_when_done = False)
    qiskit_counts = [submitter.convert_counts_to_qiskit(c) for c in counts]
    for (key, _ ) , counts in zip(experiment.qc_list.items(), qiskit_counts):
        results[key] = counts
    energy = vqe.get_energy(experiment, results, shots)

    state_vec_energy = experiment.get_energy(experiment.qc.assign_parameters(params))
    energy_differences.append(np.abs(state_vec_energy-energy))
    


average_energy_diff_per_site = np.mean(energy_differences) / (nsites*num_trials)
print('Average energy difference per-site = ', average_energy_diff_per_site)

  0%|          | 0/100 [00:00<?, ?it/s]

Average energy difference per-site =  0.01020164969521277
